## 📑 HAPS Collections Report Attribution
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYD6H)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

from pyspark.sql import DataFrame
from pyspark.sql.functions import trim


dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

In [0]:
table_name = 'HAPS_CollectionsReportAttribution'

schema = (
    "BusinessDate date,"
    "ExposureKey string,"
    "CDI string,"
    "AgreementKey string,"
    "PortfolioName string,"
    "AccountBalanceFlag string,"
    "AssetSegment string,"
    "AttributionChannel string,"
    "AttributionTeam string,"
    "AssignmentChannel int,"
    "TotalPayments double,"
    "CashPayments double,"
    "LiquidationPayments double,"
    "SoMGBVEur decimal(19,2),"
    "SoMDelinquent decimal(19,2),"
    "EoMDelinquentAmount decimal(19,2),"
    "SoMBucketGroup string,"
    "MtDBucketGroup string,"
    "TargetRate string,"
    "TargetAmount decimal(19,2),"
    "RemainBalance decimal(19,2),"
    "RollBackBalance decimal(19,2),"
    "ContractLevelAssignmentCategory string,"
    "ReportingCategoryLevel1 string,"
    "ReportingCategoryLevel2 string,"
    "ReportingCategoryLevel3 string,"
    "RPCFlag int,"
    "PositiveRPCFlag int,"
    "NegativeRPCFlag int,"
    "RestRPCFlag int,"
    "AttemptFlag int,"
    "ContactFlag int,"
    "InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""

# ================== Create Delta Table ==================
#delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"  
#AWS Credentials not available thus env_prefix returns empty temporarily we manually set it to `qa`

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date)


In [0]:
import os
from datetime import datetime
import pandas as pd
from io import BytesIO
import shutil


EOMDate = spark.sql(f"""SELECT DATE_ADD(DATE_TRUNC('month', '{run_date}'), -1)""").collect()[0][0]
RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{EOMDate}'), CAST('{EOMDate}' AS DATE)) AS DATE)").collect()[0][0]


LastDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

ReferenceDateForHistorical = spark.sql(f"""
    SELECT NextBusinessDay
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE CalendarDate = CAST(DATE_FORMAT('{run_date}', 'yyyy-MM-15') AS DATE)
""").collect()[0][0]

SoMReferenceDate = spark.sql(f"""
    SELECT PreviousBusinessDay
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE CalendarDate = '{LastDate}'
""").collect()[0][0]

ReferenceDateForHistorical = ReferenceDateForHistorical.strftime('%Y-%m-%d')
SoMReferenceDate = SoMReferenceDate.strftime('%Y-%m-%d')

## Export fields

In [0]:
collection_perimeter = spark.sql(f"""
                                    select 
                                         BusinessDate
                                        ,ExposureKey
                                        ,CDI
                                        ,AgreementKey
                                        ,PortfolioName
                                        ,AccountBalanceFlag
                                        ,AssetSegment
                                        ,SoMGBVEur
                                        ,SoMDelinquent
                                        ,EoMDelinquentAmount
                                        ,SoMBucketGroup
                                        ,MtDBucketGroup
                                        ,TargetRate
                                        ,TargetAmount
                                        ,RemainBalance
                                        ,RollBackBalance
                                        ,ContractLevelAssignmentCategory
                                        ,ReportingCategoryLevel1
                                        ,ReportingCategoryLevel2
                                        ,ReportingCategoryLevel3
                                    from `{env_prefix}-cdh-delivery`.repoint.haps_collectionsreport b
                                    where b.BusinessDate = '{run_date}'
""")

collection_perimeter.createOrReplaceTempView("collection_perimeter")

In [0]:

assigned_perimeter = spark.sql(f"""

                                select
                                b.BusinessDate,
                                b.ExposureKey,
                                b.CDI,
                                b.Agreementkey,
                                b.PortfolioName,
                                b.AccountBalanceFlag,
                                b.AssetSegment, 
                                a.AssignmentTeam,
                                b.SoMGBVEur,
                                b.SoMDelinquent,
                                b.EoMDelinquentAmount,
                                b.SoMBucketGroup,
                                b.MtDBucketGroup,
                                b.TargetRate,
                                b.TargetAmount,
                                b.RemainBalance,
                                b.RollBackBalance,
                                b.ContractLevelAssignmentCategory,
                                b.ReportingCategoryLevel1,
                                b.ReportingCategoryLevel2,
                                b.ReportingCategoryLevel3
                                from `{env_prefix}-cdh-delivery`.repoint.haps_assignmentmapping a  
                                inner join collection_perimeter b 
                                    on a.ExposureKey = b.ExposureKey 
                                where (a.AssignmentChannel = 'Outsourcing Collection' or a.CollectionIntramonthAssignment = 2) 
                                    and b.ReportingCategoryLevel3 ilike '%Collection%' 
                                    and a.BusinessDate = '{run_date}'
""")

assigned_perimeter.createOrReplaceTempView("assigned_perimeter")

In [0]:

assigned_perimeter_custom = spark.sql(f"""
                                select
                                    SUBSTRING(a.ExposureKey, 4, LEN(a.ExposureKey) - 3) AS ProcessedAccountNumber,
                                    a.*
                                from assigned_perimeter a  
""")

assigned_perimeter_custom.createOrReplaceTempView("assigned_perimeter_custom")

In [0]:

assignments = spark.sql(f"""
            select  distinct
                a.PortfolioName,
                a.ExposureKey, 
                a.AgreementKey, 
                a.CDI,
                coalesce(ChannelCode, Channel)  as AssignmentChannel,  
                Team as AssignmentTeam,
                AgentFullName as AssignmentAgent,
                {run_date!r} as RunDate,
                a.BusinessDate as AssignmentReferenceDate,
                COALESCE(DATEADD(DAY, -1, LAG(a.BusinessDate) OVER (PARTITION BY a.ExposureKey ORDER BY a.BusinessDate DESC)), a.BusinessDate) as MaxDate
            from `{env_prefix}-cdh`.primary.assignment a
            inner join assigned_perimeter_custom b
                on a.ExposureKey = b.ProcessedAccountNumber and a.CDI = b.CDI
            where   1=1
                and a.BusinessDate >= {LastDate!r} 
                and a.BusinessDate <= {run_date!r}
                and a.AssignmentType = 'Channel Assignment'

""")

assignments.createOrReplaceTempView("assignments")

In [0]:
ExchangeRates = spark.sql(f'''
  SELECT
          E.BusinessDate  
        , E.OriginalCurrency                  
        , E.ExchangeRate                                      
  FROM `{env_prefix}-cdh`.reference.exchangerates as E
  WHERE E.BusinessDate > {LastDate!r} AND E.BusinessDate <= {run_date!r}
  
''')
ExchangeRates.createOrReplaceTempView("ExchangeRates")

In [0]:
payments = spark.sql(f"""
                     SELECT 
                          regexp_replace(ExposureKey, '[A-Za-z]', '') AS AccountNumber
                        , PaymentActualDate
                        , PaymentType
                        , SUM(p.PaymentAmount/ COALESCE(e.ExchangeRate,1)) AS TotalPayments
                        ,SUM(CASE WHEN p.PaymentType IN ('Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.','Είσπραξη από πελάτη', 'INSURANCE CLAIM') THEN p.PaymentAmount/ COALESCE(e.ExchangeRate,1) ELSE 0 END) AS CashPayments 
                        , SUM(CASE WHEN p.PaymentType NOT IN ('Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.','Είσπραξη από πελάτη', 'INSURANCE CLAIM') THEN p.PaymentAmount/ COALESCE(e.ExchangeRate,1) ELSE 0 END) AS Liquidations     
                        , CASE WHEN c.IsBusinessDay = 0 THEN c.PreviousBusinessDay ELSE c.CalendarDate END AS PaymentDateBusiness  
                    FROM `{env_prefix}-cdh`.primary.payment p
                    INNER JOIN `{env_prefix}-cdh`.primary.portfolio por
                    ON por.unit_code_profits=p.portfolioKey and p.BusinessDate=por.BusinessDate
                    LEFT JOIN ExchangeRates e
                        ON p.Currency = e.OriginalCurrency AND p.BusinessDate = e.BusinessDate
                    LEFT JOIN `{env_prefix}-cdh`.reference.ref_calendar AS c ON p.PaymentActualDate = c.CalendarDate 
                    WHERE p.BusinessDate={run_date!r}                           
                      AND p.PaymentActualDate > LAST_DAY(DATEADD(MONTH, -1, {run_date!r}))		
                      AND p.PaymentActualDate <= {run_date!r} 
                      AND p.SourceSystem='profits'
                    GROUP BY ExposureKey, PaymentActualDate, PaymentType
                    , CASE WHEN c.IsBusinessDay = 0 THEN c.PreviousBusinessDay ELSE c.CalendarDate END  
                    
""")

payments.createOrReplaceTempView("payments")

In [0]:
all_collections = spark.sql(f"""
            with grouped_payments(
                SELECT DISTINCT         
                      a.*,
                      p.PaymentActualDate, 
                      p.TotalPayments,
                      p.CashPayments, 
                      p.Liquidations AS LiquidationPayments
                FROM assignments a   
                LEFT JOIN payments p
                    ON a.ExposureKey = p.AccountNumber AND a.AssignmentReferenceDate = p.PaymentDateBusiness 
            )
            select  PortfolioName,
                ExposureKey, 
                AgreementKey, 
                CDI,
                AssignmentChannel,  
                AssignmentTeam,
                AssignmentAgent,
                SUM(TotalPayments) AS TotalPayments,
                SUM(CashPayments) AS CashPayments,
                SUM(LiquidationPayments) AS LiquidationPayments
            from grouped_payments
            group by PortfolioName,
                    ExposureKey, 
                    AgreementKey, 
                    CDI,
                    AssignmentChannel,  
                    AssignmentTeam,
                    AssignmentAgent 
""")

all_collections.createOrReplaceTempView("all_collections")

In [0]:
Interactions_v0 = spark.sql(f'''

        SELECT DISTINCT
            ia.ExposureKey  
        , ia.PortfolioName 
        , ia.CDI 
        , ia.InteractionTypeKey
        , ia.InteractionDate
        , ia.InteractionType 
        , ia.InteractionKey
        , ia.CreatedDate  
        , rank () over (partition by ia.ExposureKey, ia.InteractionKey order by ia.businessdate asc) as rnk
        FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
        INNER JOIN assigned_perimeter_custom b
            ON ia.ExposureKey = b.ProcessedAccountNumber and ia.CDI = b.CDI
        WHERE  1=1
            and InteractionDate >= {LastDate!r}
            and InteractionDate <= {run_date!r}
            and SourceSystem = 'qca'
''')
Interactions_v0.createOrReplaceTempView("Interactions_v0")

In [0]:
InteractionsAll = spark.sql(f'''

        SELECT DISTINCT
                  ia.ExposureKey
                , ia.PortfolioName
                , ia.CDI
                , ia.InteractionTypeKey
                , ia.InteractionDate
                , ia.InteractionType
                , ia.CreatedDate 
                , ia.cdi
                , m.Activity_Code
                , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 4    
                        WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
                        WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
                        WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
                        WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
                        WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
                END AS Priority
                , CASE WHEN left(m.`1st_level_category`,1) = '5'             THEN 5    
                        WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
                        WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
                        WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
                        WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
                        WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
                END AS Priority_Aux    
                , m.`1st_level_category`
                , m.`2nd_level_category`
                , m.hierarchy
                , m.`3rd_level_category`
                , M.base_category
                , rank() over (partition by ia.ExposureKey, m.Activity_Group, ia.InteractionDate order by
                                CASE  WHEN left(m.`1st_level_category`,1) = '5'             THEN 4    
                                WHEN left(m.`1st_level_category`,1) = '4'             THEN 4
                                WHEN left(m.`1st_level_category`,1) = '3'             THEN 3
                                WHEN left(m.`1st_level_category`,1) = '2'             THEN 2                      
                                WHEN left(m.`2nd_level_category`,3) = '1.2'           THEN 1.2
                                WHEN left(m.`2nd_level_category`,3) = '1.1'           THEN 1.1
                                        END desc
                                        , ia.CreatedDate desc
                                        , ia.InteractionKey desc) as rank
                , m.Activity_Group
                , CASE WHEN m.`1st_level_category` IN ('1. no contact', '2. contact - Not RPC', '5.Positive Communication') THEN m.`3rd_level_category`
                        WHEN m.`1st_level_category` = '3. Rest RPC Communications' THEN m.`1st_level_category`
                        WHEN m.`1st_level_category` = '4. Negative Communication' THEN m.`2nd_level_category`      
                END AS ActivityTypeGroup
        FROM Interactions_v0 ia
        JOIN `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia.InteractionTypeKey = m.Activity_Code     
        WHERE  (m.hierarchy  not in (0,1) or (m.hierarchy = 0 and base_category = 'attempt'))
            AND ia.rnk = 1
            AND m.BusinessDate = (select max(BusinessDate) from `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping)

'''
)
InteractionsAll.createOrReplaceTempView("InteractionsAll")


In [0]:
GroupedLoanActivities = spark.sql(f'''

    SELECT DISTINCT   
        PortfolioName
      , ExposureKey
      , CDI
      --, ia.Activity_Group
      --, ia.InteractionDate
      , MAX(CASE WHEN IA.Priority_Aux in (3,4,5) THEN 1 ELSE 0 END) AS RPCFlag
      , MAX(CASE WHEN IA.Priority_Aux = 5 THEN 1 ELSE 0 END) AS PositiveRPCFlag
      , MAX(CASE WHEN IA.Priority_Aux = 4 THEN 1 ELSE 0 END) AS NegativeRPCFlag
      , MAX(CASE WHEN IA.Priority_Aux = 3 THEN 1 ELSE 0 END) AS RestRPCFlag
      , MAX(CASE WHEN IA.Priority_Aux >= 1.2 THEN 1 ELSE 0 END) AS AttemptFlag
      , MAX(CASE WHEN IA.Priority_Aux >= 2 THEN 1 ELSE 0 END) AS ContactFlag
    FROM InteractionsAll AS iA 
    GROUP BY  PortfolioName
            , ExposureKey
            , CDI
    --, ia.InteractionDate , IA.Activity_Group

'''
)
GroupedLoanActivities.createOrReplaceTempView("GroupedLoanActivities")


In [0]:
export_perimeter = spark.sql(f"""
                                select distinct
                                    a.BusinessDate,
                                    a.ExposureKey,
                                    a.CDI,
                                    a.AgreementKey,
                                    a.PortfolioName,
                                    a.AccountBalanceFlag,
                                    a.AssetSegment, 
                                    ac.AssignmentChannel as AttributionChannel, 
                                    ac.AssignmentTeam as AttributionTeam, 
                                    case when acr.ExposureKey is not null then 1 else 0 end as AssignmentChannel,
                                    ac.TotalPayments,
                                    ac.CashPayments, 
                                    ac.LiquidationPayments, 
                                    a.SoMGBVEur,
                                    a.SoMDelinquent,
                                    a.EoMDelinquentAmount,
                                    a.SoMBucketGroup,
                                    a.MtDBucketGroup,
                                    a.TargetRate,
                                    a.TargetAmount,
                                    a.RemainBalance,
                                    a.RollBackBalance,
                                    a.ContractLevelAssignmentCategory,
                                    a.ReportingCategoryLevel1,
                                    a.ReportingCategoryLevel2,
                                    a.ReportingCategoryLevel3,
                                    intr.RPCFlag,
                                    intr.PositiveRPCFlag,
                                    intr.NegativeRPCFlag,
                                    intr.RestRPCFlag,
                                    intr.AttemptFlag,
                                    intr.ContactFlag
                                from assigned_perimeter_custom a
                                left join all_collections ac
                                    on a.ProcessedAccountNumber = ac.ExposureKey and a.CDI = ac.CDI 
                                left join all_collections acr
                                    on a.ProcessedAccountNumber = acr.ExposureKey and a.CDI = ac.CDI and a.AssignmentTeam = ac.AssignmentTeam
                                left join GroupedLoanActivities intr
                                    on a.ProcessedAccountNumber = intr.ExposureKey and a.CDI = intr.CDI 
                                    
""")

export_perimeter.createOrReplaceTempView("export_perimeter")

In [0]:
final_df = spark.sql(
    f'''
    SELECT 
        CAST(BusinessDate AS DATE) AS BusinessDate,
        CAST(ExposureKey AS STRING) AS ExposureKey,
        CAST(CDI AS STRING) AS CDI,
        CAST(AgreementKey AS STRING) AS AgreementKey,
        CAST(PortfolioName AS STRING) AS PortfolioName,
        CAST(AccountBalanceFlag AS STRING) AS AccountBalanceFlag,
        CAST(AssetSegment AS STRING) AS AssetSegment,
        CAST(AttributionChannel AS STRING) AS AttributionChannel,
        CAST(AttributionTeam AS STRING) AS AttributionTeam,
        CAST(AssignmentChannel AS INT) AS AssignmentChannel,
        CAST(TotalPayments AS DOUBLE) AS TotalPayments,
        CAST(CashPayments AS DOUBLE) AS CashPayments,
        CAST(LiquidationPayments AS DOUBLE) AS LiquidationPayments,
        CAST(SoMGBVEur AS DECIMAL(19,2)) AS SoMGBVEur,
        CAST(SoMDelinquent AS DECIMAL(19,2)) AS SoMDelinquent,
        CAST(EoMDelinquentAmount AS DECIMAL(19,2)) AS EoMDelinquentAmount,
        CAST(SoMBucketGroup AS STRING) AS SoMBucketGroup,
        CAST(MtDBucketGroup AS STRING) AS MtDBucketGroup,
        CAST(TargetRate AS STRING) AS TargetRate,
        CAST(TargetAmount AS DECIMAL(19,2)) AS TargetAmount,
        CAST(RemainBalance AS DECIMAL(19,2)) AS RemainBalance,
        CAST(RollBackBalance AS DECIMAL(19,2)) AS RollBackBalance,
        CAST(ContractLevelAssignmentCategory AS STRING) AS ContractLevelAssignmentCategory,
        CAST(ReportingCategoryLevel1 AS STRING) AS ReportingCategoryLevel1,
        CAST(ReportingCategoryLevel2 AS STRING) AS ReportingCategoryLevel2,
        CAST(ReportingCategoryLevel3 AS STRING) AS ReportingCategoryLevel3,
        CAST(RPCFlag AS INT) AS RPCFlag,
        CAST(PositiveRPCFlag AS INT) AS PositiveRPCFlag,
        CAST(NegativeRPCFlag AS INT) AS NegativeRPCFlag,
        CAST(RestRPCFlag AS INT) AS RestRPCFlag,
        CAST(AttemptFlag AS INT) AS AttemptFlag,
        CAST(ContactFlag AS INT) AS ContactFlag,
        CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
    FROM export_perimeter
    '''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path)

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
CollectionsReportAttribution = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
CollectionsReportAttribution.createOrReplaceTempView("CollectionsReportAttribution")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/CollectionsReportAttribution/" 
file_name = "CollectionsReportAttribution_" +run_date+ ".csv"
CollectionsReportAttribution.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") 
rename_and_cleanup_files(output_path, file_name)